## Step 1: Import Libraries & Load Preprocessed Data

In [7]:
import pandas as pd
import numpy as np
import pickle
import datetime
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_selection import SelectKBest, chi2

# Load preprocessed data
dataset1 = pd.read_csv('fraud_preprocessed.csv', index_col=None)
df2 = dataset1.copy()
df2 = pd.get_dummies(df2, drop_first=True)

indep_X = df2.drop('Fraud_Label', axis=1)
dep_Y   = df2['Fraud_Label']

print('✅ Libraries imported!')
print('Feature shape:', indep_X.shape)
print('Target shape :', dep_Y.shape)

✅ Libraries imported!
Feature shape: (50000, 22)
Target shape : (50000,)


## Step 2: SelectKBest Feature Selection (Top 3 Features)

In [8]:
def SelectKBest_features(indep, dep, n):
    from sklearn.feature_selection import SelectKBest, chi2
    test             = SelectKBest(score_func=chi2, k=n)
    fit              = test.fit(indep, dep)
    selectk_features = fit.transform(indep)
    feature_names    = indep.columns[fit.get_support(indices=True)].tolist()
    return selectk_features, feature_names

kbest, Feature_Names = SelectKBest_features(indep_X, dep_Y, 3)
print('✅ Top 3 Features selected:')
print(Feature_Names)

# Train-Test split on selected features
X_train, X_test, y_train, y_test = train_test_split(
    kbest, dep_Y, test_size=0.25, random_state=0)

# Scale
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test  = sc.transform(X_test)

print('Train size:', X_train.shape[0], '| Test size:', X_test.shape[0])

✅ Top 3 Features selected:
['Account_Balance', 'Failed_Transaction_Count_7d', 'Risk_Score']
Train size: 37500 | Test size: 12500


## Step 3: Train Final Model — Random Forest

In [9]:
final_model = RandomForestClassifier(
    n_estimators = 100,
    criterion    = 'entropy',
    max_depth    = None,
    random_state = 0,
    n_jobs       = -1
)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)

print('Final Model Trained!')
print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')

Final Model Trained!
Accuracy : 1.0000


In [14]:
Save Final Model (Correctly)
import pickle

# Save actual trained model (not a string!)
filename = 'Fraud_Prediction_final_model.sav'
pickle.dump(final_model, open(filename, 'wb'))


In [11]:
# Get User Input
# User enters transaction details
Transaction_amount = float(input('Enter the transaction amount:'))
Failed_Txn_Count   = int(input('Enter the failed transaction count:'))
Risk_Score_val     = float(input('Enter the risk score:'))

# Scale the input using same scaler
preinput = sc.transform([[Transaction_amount, Failed_Txn_Count, Risk_Score_val]])

print('Input captured!')
print(f'   Amount      : Rs.{Transaction_amount}')
print(f'   Failed Txns : {Failed_Txn_Count}')
print(f'   Risk Score  : {Risk_Score_val}')

Enter the transaction amount: 200000
Enter the failed transaction count: 2
Enter the risk score: 5



✅ Input captured!
   Amount      : Rs.200000.0
   Failed Txns : 2
   Risk Score  : 5.0


## Step 6: Load Saved Model & Predict

In [12]:
import pickle

# Load the saved model
loaded_model      = pickle.load(open('Fraud_Prediction_final_model.sav', 'rb'))

# Predict
future_prediction = loaded_model.predict(preinput)

print('Prediction:', future_prediction[0],
      '→ FRAUD' if future_prediction[0]==1 else '→ GENUINE')

Prediction: 1 → FRAUD


In [13]:
#Show Result Message
import datetime
now = datetime.datetime.now()

if future_prediction == 1:
    print(f'The transaction dated {now} for the amount Rs.{Transaction_amount}'
          f' initiated by you is found to be suspicious and we request you'
          f' to kindly contact the bank if it is not initiated by you.')
else:
    print('The transaction is found to be genuine')

The transaction dated 2026-05-12 12:36:04.611262 for the amount Rs.200000.0 initiated by you is found to be suspicious and we request you to kindly contact the bank if it is not initiated by you.
